# **Import Necessary Packages**

In [ ]:
!pip install meteostat
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd
from meteostat import Point, Daily
from pathlib import Path
from scipy import stats
import numpy as np

# **Import & Load Data**

In [ ]:
# Step 1: Load the crime dataset
# Think of this as opening the city’s crime logbook. Each row is a story of an incident in Charlottesville.
crime_df = pd.read_csv("Crime_Data.csv")

In [ ]:
# Step 2: Retrieve weather data
# Now we bring in the other half of the story: the temperature.
# We’ll align the weather timeline with the crime reports so we can see how they move together.
oldest_date = datetime.strptime(crime_df['DateReported'].min(), '%Y/%m/%d %H:%M:%S+00')
newest_date = datetime.strptime(crime_df['DateReported'].max(), '%Y/%m/%d %H:%M:%S+00')

# COMPLETE: Retrieve Charlottesville's coordinates and update them below
charlottesville = Point(______, ______)
data = Daily(charlottesville, oldest_date, newest_date)
weather_data = data.fetch()

# **Data Cleaning & Preprocesing**

In [ ]:
# Step 3: Clean and prepare the data
crime_df['DateReported'] = pd.to_datetime(crime_df['DateReported'])
crime_by_date = crime_df.groupby(crime_df['DateReported'].dt.date).size().reset_index(name='Crime Count')
crime_by_date.rename(columns={'DateReported': 'Date'}, inplace=True)

weather_data = weather_data.reset_index()
weather_data['time'] = pd.to_datetime(weather_data['time'])
crime_by_date['Date'] = pd.to_datetime(crime_by_date['Date'])
weather_data['time'] = weather_data['time'].dt.date
weather_data['time'] = pd.to_datetime(weather_data['time'])

In [ ]:
# Step 4: Merge datasets
merged_data = pd.merge(crime_by_date, weather_data, left_on='Date', right_on='time', how='inner')

# **Data Visualization**

In [ ]:
# Step 5: Explore offense types
# Let’s start by asking: what kinds of crimes are most common here?
# COMPLETE: Generate a bar graph of all crime offense types, using matplotlib or seaborn!

In [ ]:
# Step 6: Visualize the relationship
# This line graph lets you ride along the timeline of Charlottesville, watching crime counts and average temperature rise and fall together.
# COMPLETE: Generate a line graph of crime count and average temperature over time in Charlottesville, using matplotlib or seaborn!

# **Calculting the Correlation between Crime and Temperature in Charlottesville**

In [ ]:
# Step 7: Test the big-picture relationship
# Now that crime and temperature are merged, let’s ask the key question:
# Do hotter days really line up with more crime overall?

# COMPLETE: Your turn to work and fill in the blanks below!
# 1) Create a dataframe that contains only the columns you need for correlation. Hint: Remember we're looking at crime and temperature.
# 2) Drop any missing values so the test runs cleanly.
# 3) Use stats.pearsonr() to calculate the correlation coefficient (r) and p-value.

corr_df = _______
r, p_two_sided = stats.pearsonr(______, ______)
p_one_sided = ______ if r > 0 else 1.0

# Report your results!
# Pearson r tells us the strength of the relationship (closer to 1 = stronger positive correlation).
# The p-values tell us whether the relationship is statistically significant (r ≥ 0.5, p < 0.05) or not.

print(f"[TOTAL] Pearson r = {r:.3f}")
print(f"[TOTAL] Two-sided p-value = {p_two_sided:.4g}")
print(f"[TOTAL] One-sided p (H1: r > 0) = {p_one_sided:.4g}")

In [ ]:
# Step 8: Break crime down by type
# Instead of looking at crime as one big number, let’s zoom in.
# Which offenses are most common, and do they show stronger links to temperature?

# COMPLETE: Again, work through this guided by the blanks and convert 'DateReported' to datetime, then group by Date and Offense.

crime_df['DateReported'] = pd.to_datetime(crime_df['DateReported'], errors='coerce')
daily_counts = (
    crime_df
    .assign(Date=_____)
    .groupby([_____, _____])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
daily_counts['Date'] = pd.to_datetime(daily_counts['Date'])

In [ ]:
# Step 9: Focus on the top 12 offenses
# COMPLETE: use the head() function

total_offense_counts = crime_df['Offense'].value_counts()
top12 = _____
daily_top12 = daily_counts[['Date'] + list(top12)].copy()

# Question: Why do we focus on the top 12 instead of all 100+ offenses?
# Write your reasoning in a markdown cell before continuing.
# [Insert response here]

In [ ]:
# Step 10: Add temperature back in
# COMPLETE: Merge the crime counts with weather data.

wx_cols = weather_data[['time', 'tavg']].rename(columns={'time': 'Date'})
wx_cols['Date'] = pd.to_datetime(wx_cols['Date'])
merged_top12 = pd.merge(_____, _____, on='Date', how='inner').dropna(subset=['tavg'])

In [ ]:
# Step 11: Correlation for each offense
# COMPLETE: Loop through each of the 12 offenses and calculate Pearson r.
# This shows us whether certain crimes (like assaults or thefts) rise more sharply with the heat.

# Which offenses do you expect to be most correlated with temperature? Why?
# [Insert response here]

rows = []
for offense in top12:
    x = merged_top12['tavg']
    y = merged_top12[offense]
    if y.nunique() < 2:
        rows.append((offense, np.nan, np.nan))
        continue
    r_off, p2 = stats.pearsonr(_____, _____)
    p1 = _____ if r_off > 0 else 1.0
    rows.append((offense, r_off, p1))

# **Conclusion**

In [ ]:
# Step 12: Summarize the findings
# COMPLETE: Which offenses show the strongest correlation with temperature?
offense_corr = pd.DataFrame(rows, columns=['Offense', 'Pearson_r', 'OneSided_p'])
offense_corr = offense_corr.sort_values(by='Pearson_r', ascending=False)

print("\n[TOP 12 OFFENSES] Pearson correlation with temperature (one-sided p for r>0):")
print(offense_corr.to_string(index=False))

# Reflection:
# After you compute the correlations, compare them to your predictions.
# Were you surprised? Which offenses showed the strongest link to heat?
# [Insert response here]